# Modern CNNs
Today we will cover:
- Theory: ResNet, EfficientNet
- Exercise: Fine-tune ResNet on custom dataset
- Mini Project: Transfer learning for flower classification

## Why Study Modern CNNs?
Deep CNNs evolved to solve challenges of depth, efficiency, and scalability.

### ResNet (2015)
- Introduced Residual Connections (skip connections).
- Solves vanishing gradient problem in very deep networks.
- Achieved state-of-the-art on ImageNet.
- Key Idea: Learn residuals instead of direct mappings.

### EfficientNet (2019)
- Scales depth, width, and resolution systematically.
- Uses compound scaling for balanced efficiency.
- Achieves high accuracy with fewer parameters.
- Key Idea: Efficient scaling across dimensions.

## Setup & Custom Dataset Loader

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets

# Example: Custom dataset path (replace with your dataset folder)
data_dir = './custom_data'

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

trainset = datasets.ImageFolder(root=data_dir+'/train', transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True)

testset = datasets.ImageFolder(root=data_dir+'/test', transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=32, shuffle=False)

class_names = trainset.classes
print("Classes:", class_names)

## Exercise: Fine-tune ResNet on Custom Dataset

In [ ]:
import torchvision.models as models

# Load pretrained ResNet18
resnet18 = models.resnet18(pretrained=True)

# Freeze early layers
for param in resnet18.parameters():
    param.requires_grad = False

# Replace final layer for custom classes
num_ftrs = resnet18.fc.in_features
resnet18.fc = nn.Linear(num_ftrs, len(class_names))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet18.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet18.fc.parameters(), lr=0.001)

# Training loop (simplified)
for epoch in range(1, 3):
    running_loss = 0.0
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = resnet18(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch}, Loss: {running_loss/len(trainloader):.4f}")

## Mini Project: Transfer Learning for Flower Classification

In [ ]:
# Evaluate on test set
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = resnet18(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy on test set: {100 * correct / total:.2f}%")

# Predict on a batch of flower images
import matplotlib.pyplot as plt
import numpy as np

def imshow(img):
    img = img / 2 + 0.5
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

dataiter = iter(testloader)
images, labels = next(dataiter)
imshow(torchvision.utils.make_grid(images[:4]))
outputs = resnet18(images.to(device))
_, predicted = torch.max(outputs, 1)
print('Predicted:', ' '.join(f'{class_names[predicted[j]]}' for j in range(4)))